# Flow matching on toy 2D data

Before training a diffusion transformer on images, this notebook builds the intuition on data you can
look at directly: points on two concentric circles. Everything runs on CPU in minutes.

Two questions drive it:

1. A flow-matching model can be trained to predict the clean data **x** or the velocity **v**. Does it matter?
2. Real diffusion models work in high-dimensional spaces (a 256x256 image, or a 16x16x32 latent). What
   happens to each choice when the *ambient* dimension grows while the data itself stays 2D?

The answer to the second question is what motivates `transport.prediction` in the MiniDog configs.

## Data

The dataset is 2D circle points, plus the same points multiplied by a random orthogonal matrix
`P` of shape `(D, 2)` for `D = 8` and `32`. An orthogonal projection keeps distances, so the
data is still a 2D ring, just embedded in D dimensions. `to_2d` undoes the projection so we can
plot samples from any D on the same 2D axes.

In [ ]:
!uv run hf download xingjianleng/toy-data --local-dir data/toy_data --repo-type dataset --include circles.npz

In [ ]:
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


class ToyDiffusionDataset(Dataset):
    def __init__(
        self, data_dir: Path, dim: int = 2,
    ):
        root = Path(data_dir)
        npz = np.load(root / "circles.npz")

        self.data = torch.from_numpy(npz[f"{dim}d"])
        self.dim = dim
        self.P = npz[f"P_{dim}"] if dim > 2 else None

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

    def to_2d(self, samples: np.ndarray) -> np.ndarray:
        if self.P is not None:
            return samples @ self.P.T
        return samples


def get_dataloader(
    data_dir: Path, dim: int = 2, batch_size: int = 1024, shuffle: bool = True,
) -> DataLoader:
    ds = ToyDiffusionDataset(data_dir=data_dir, dim=dim)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)

## Model

A plain MLP takes the noisy point `x_t` and the time `t` and outputs a vector of the same size as
the data. Time is fed in as a sinusoidal embedding: a fixed set of sine and cosine waves of `t` at
different frequencies, the same trick a DiT uses for its timestep. `hidden_dim` is the width;
Experiment 2 sweeps it.

In [ ]:
import math
import torch
import torch.nn as nn


class SinusoidalEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        emb = math.log(10_000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=t.device) * -emb)
        emb = t * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class MLPDenoiser(nn.Module):
    def __init__(
        self, data_dim: int, hidden_dim: int = 256, n_layers: int = 5, time_dim: int = 128
    ):
        super().__init__()
        self.time_embed = SinusoidalEmbedding(time_dim)

        layers = []
        in_dim = data_dim + time_dim
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.ReLU())
            in_dim = hidden_dim
        layers.append(nn.Linear(hidden_dim, data_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        return self.net(torch.cat([x, t_emb], dim=-1))

## Flow matching

Corrupt clean data `x` toward Gaussian noise `eps` along a straight line:

$$x_t = (1 - t)\,x + t\,\epsilon, \qquad t \in [0, 1]$$

so `t = 0` is data and `t = 1` is pure noise. The velocity of that line is constant,
`v = eps - x`, and that is what the model needs to learn: given `x_t` and `t`, output `v`.

The model can parameterize this in two ways. Both are trained with the same MSE on `v`:

- **v-prediction**: the network outputs `v` directly.
- **x-prediction**: the network outputs `x`, and we convert with `v = (x_t - x) / t`.

The conversion divides by `t`, which blows up near `t = 0`, so `t` is clipped to `[T_EPS, 1 - T_EPS]`.

**Sampling** starts from noise at `t = 1` and walks the line back to `t = 0` with Euler steps
`x_t <- x_t + v * dt` (`dt` is negative), converting the model's output to `v` at every step.

In [ ]:
import torch
import torch.nn.functional as F
from tqdm import tqdm

T_EPS = 1e-2


class FlowMatching:
    def __init__(self, sample_steps: int = 50):
        self.sample_steps = sample_steps

    def training_losses(self, model, x_0, pred_type):
        B = x_0.shape[0]
        t = torch.rand((B, 1), device=x_0.device).clip(T_EPS, 1 - T_EPS)
        eps = torch.randn_like(x_0)
        xt = (1 - t) * x_0 + t * eps

        pred_raw = model(xt, t)

        # v = eps - x  (velocity from data toward noise)
        if pred_type == "x":
            v_hat = (xt - pred_raw) / t
        elif pred_type == "v":
            v_hat = pred_raw
        target = eps - x_0
        return F.mse_loss(v_hat, target)

    @torch.no_grad()
    def sample(self, model, shape, pred_type, device="cpu"):
        x_t = torch.randn(shape, device=device)  # start at t=1 (noise)
        N = shape[0]
        ts = torch.linspace(1 - T_EPS, T_EPS, self.sample_steps + 1, device=device)

        for i in tqdm(range(self.sample_steps), desc="sampling", leave=False):
            t = ts[i].expand(N, 1)
            dt = ts[i + 1] - ts[i]  # negative (going from t=1 to t=0)
            pred_raw = model(x_t, t)

            if pred_type == "x":
                v_hat = (x_t - pred_raw) / t
            elif pred_type == "v":
                v_hat = pred_raw

            x_t = x_t + v_hat * dt

        return x_t

## Training loop

Adam, a fixed number of steps, batches drawn from the dataset with reshuffling. `train_one` returns
the trained model; the choice of `pred_type` and `dim` are the two knobs the experiments turn.

In [ ]:
from tqdm import trange


device = "cuda" if torch.cuda.is_available() else "cpu"


def train_one(
    pred_type: str,
    data_dir: Path = Path("data/toy_data"),
    dim: int = 2,
    train_steps: int = 25000,
    batch_size: int = 1024,
    lr: float = 2e-3,
    hidden_dim: int = 256,
    n_layers: int = 5,
) -> nn.Module:
    print(f"Training: {pred_type}_pred (dim={dim})")

    dl = get_dataloader(data_dir=data_dir, dim=dim, batch_size=batch_size)
    flow = FlowMatching()
    model = MLPDenoiser(dim, hidden_dim=hidden_dim, n_layers=n_layers).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    pbar = trange(train_steps)
    step = 0
    while step < train_steps:
        for batch in dl:
            if step >= train_steps:
                break
            x1 = batch.to(device)
            loss = flow.training_losses(model, x1, pred_type)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            step += 1
            pbar.update(1)
    pbar.close()

    return model


## Experiment 1: x-prediction vs v-prediction as the dimension grows

Same model, same steps, for `D = 2` and `D = 32`. Each row shows the ground truth and samples from
both parameterizations, projected back to 2D.

What to look for: at `D = 2` both should recover the rings. At `D = 32` the data is still a 2D ring,
but the noise `eps` now lives in 32 dimensions. A v-prediction network must output all 32
components of `eps - x`, most of which are pure noise it cannot predict; an x-prediction network
only has to find the 2D manifold the data sits on. Check whether the v-pred panel at `D = 32`
degrades while x-pred holds.

In [ ]:
import matplotlib.pyplot as plt

DIMS, TRAIN_STEPS, N_SAMPLES = [2, 32], 25000, 2000
SETUPS = ["x", "v"]
flow = FlowMatching(sample_steps=50)

fig, axes = plt.subplots(len(DIMS), 1 + len(SETUPS), figsize=(17, 3.7 * len(DIMS)))
for row, dim in zip(axes, DIMS):
    ds = ToyDiffusionDataset(Path("data/toy_data"), dim=dim)
    gt = ds.to_2d(ds.data.numpy()[:N_SAMPLES])
    row[0].scatter(gt[:, 0], gt[:, 1], s=2, alpha=0.4, c="black")
    row[0].set_title("Ground truth"); row[0].set_ylabel(f"D = {dim}", fontsize=13)
    lim = 1.15 * abs(gt).max()
    for ax, pred_type in zip(row[1:], SETUPS):
        model = train_one(pred_type, dim=dim, train_steps=TRAIN_STEPS)
        samples = ds.to_2d(flow.sample(model, (N_SAMPLES, dim), pred_type, device=device).cpu().numpy())
        ax.scatter(samples[:, 0], samples[:, 1], s=2, alpha=0.4, c="#2563eb")
        ax.set_title(f"{pred_type}-pred")
    for ax in row:
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


## Experiment 2: can a wider network rescue v-prediction?

The RAE paper makes a related observation for diffusion in high-dimensional latent spaces: the
denoiser's width has to grow with the dimension of the space it works in. Here we test that on the
failing case, v-prediction at `D = 32`, by sweeping `hidden_dim` from 128 to 1024 with everything
else fixed.

If the observation transfers, the rings should reappear as the width increases. Compare the cost:
x-prediction got the same result at the default width.

In [ ]:
import matplotlib.pyplot as plt

DIM, TRAIN_STEPS, N_SAMPLES = 32, 25000, 2000
HIDDEN_DIMS = [128, 256, 512, 1024]
flow = FlowMatching(sample_steps=50)

ds = ToyDiffusionDataset(Path("data/toy_data"), dim=DIM)
gt = ds.to_2d(ds.data.numpy()[:N_SAMPLES])
lim = 1.15 * abs(gt).max()

fig, axes = plt.subplots(1, 1 + len(HIDDEN_DIMS), figsize=(3.4 * (1 + len(HIDDEN_DIMS)), 3.7))
axes[0].scatter(gt[:, 0], gt[:, 1], s=2, alpha=0.4, c="black")
axes[0].set_title("Ground truth"); axes[0].set_ylabel(f"D = {DIM}, v-pred", fontsize=13)
for ax, hidden_dim in zip(axes[1:], HIDDEN_DIMS):
    model = train_one("v", dim=DIM, train_steps=TRAIN_STEPS, hidden_dim=hidden_dim)
    samples = ds.to_2d(flow.sample(model, (N_SAMPLES, DIM), "v", device=device).cpu().numpy())
    ax.scatter(samples[:, 0], samples[:, 1], s=2, alpha=0.4, c="#2563eb")
    ax.set_title(f"hidden_dim={hidden_dim}")
for ax in axes:
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()
